# Phase 1: reproduction of "An Optimized Ensemble Model for Predicting Average Localization Error of WSNs"

Base paper: Nti, Rout, Yeboah, *Decision Analytics Journal* 12 (2024) 100510.
Random Forest tuned with Grid Search, Random Search and Bayesian Optimization on the UCI ALE dataset (107 rows).
The code lives in `src/` and `scripts/`; this notebook runs the same pipeline with the same seed.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# make `src/` importable whether the notebook is opened from the repo root or from notebooks/
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.config import load_config, out_dir
from src.data import FEATURES, TARGET, load_data, split_data
from src.metrics import METRIC_NAMES
from src.paper_reference import (CORR_WITH_ALE, FEATURE_IMPORTANCE, OPTIMIZER_LABELS, TABLE1)
from src.pipeline import (comparison_table, evaluate_all, feature_importances, metrics_table,
                          params_comparison_table, save_phase1, select_by_cv, tune_all, tuning_table)
from src.plots import (plot_correlation, plot_distributions, plot_importance,
                       plot_metric_comparison, plot_pred_vs_actual)

pd.set_option("display.precision", 4)
cfg = load_config()
SEED = cfg["seed"]
np.random.seed(SEED)
FIG = out_dir(cfg, "figures")
print("Seed:", SEED)

## 1. Problem

In a wireless sensor network only a few nodes (anchors) know their position; the rest estimate it. The mean distance between estimated and true position is the Average Localization Error (ALE, meters). Inputs:

| feature | meaning |
|---|---|
| `anchor_ratio` | share of nodes that know their position |
| `trans_range` | radio range of a node (m) |
| `node_density` | number of deployed nodes |
| `iterations` | rounds of the localization algorithm |

The paper predicts ALE from these four settings with a Random Forest and compares three ways of tuning it.

## 2. Data

UCI dataset 844 (Singh et al., 2020). The loader keeps the four features and ALE and drops the std-of-ALE column.

In [ ]:
df = load_data(cfg)
print("Shape:", df.shape, "  (paper: 107 rows, 4 features + target)")
print("Missing values:", int(df.isna().sum().sum()), "  Duplicate rows:", int(df.duplicated().sum()))
display(df.head())
desc = df.describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
display(desc)

Comparison of summary statistics with the paper's Table 1:

In [ ]:
rows = []
for col, stats in TABLE1.items():
    for stat, paper_val in stats.items():
        ours = float(desc.loc[col, stat])
        rows.append({"variable": col, "statistic": stat, "paper": paper_val, "ours": round(ours, 4),
                     "match": bool(np.isclose(ours, paper_val, rtol=0.01, atol=0.011))})
chk = pd.DataFrame(rows)
print(f"{int(chk['match'].sum())}/{len(chk)} statistics match the paper's Table 1")
display(chk[~chk["match"]] if (~chk["match"]).any() else "All statistics match.")

In [ ]:
fig1 = plot_distributions(df, FIG / "eda_distributions.png")
fig2 = plot_correlation(df, FIG / "eda_correlation.png")
corr = df.corr()[TARGET].drop(TARGET)
display(pd.DataFrame({"paper": pd.Series(CORR_WITH_ALE), "ours": corr}).loc[FEATURES])
plt.show()

Node density and iterations have the strongest negative correlation with ALE.

## 3. Split and preprocessing

80/20 split as in the paper, Min-Max scaling fitted inside the pipeline (train data only). The test set has about 22 rows.

In [ ]:
X_train, X_test, y_train, y_test = split_data(df, cfg, SEED)
print(f"train: {len(X_train)} rows, test: {len(X_test)} rows")

## 4. Baseline Random Forest (default settings)

In [ ]:
from src.pipeline import evaluate_params
base_metrics, base_model, base_pred = evaluate_params({}, X_train, y_train, X_test, y_test, SEED)
display(pd.DataFrame([base_metrics], index=["Baseline RF"]))

## 5. Hyperparameter tuning

Tuned: `n_estimators`, `max_depth`, `min_samples_split`, `min_samples_leaf`. 5-fold CV on the training set.
Grid Search tries the full grid (400 combinations); Random Search and Bayesian Optimization (Optuna TPE) get 50 trials each.

In [ ]:
tuned = tune_all(X_train, y_train, cfg, SEED)
display(tuning_table(tuned))
print("Best optimizer by cross-validation score:", OPTIMIZER_LABELS[select_by_cv(tuned)])

## 6. Test-set results

Each tuned forest is refitted on the training set and scored once on the test set. MAPE is a fraction.

In [ ]:
metrics, models, preds = evaluate_all(tuned, X_train, y_train, X_test, y_test, SEED)
display(metrics_table(metrics))
plot_pred_vs_actual(y_test.to_numpy(), preds, metrics, FIG / "pred_vs_actual.png")
plt.show()

## 7. Feature importance

Impurity-based importance of the optimizer selected by CV score, next to the paper's Fig. 5.

In [ ]:
imp = feature_importances(models)
selected = select_by_cv(tuned)
display(imp.loc[list(FEATURE_IMPORTANCE)].assign(paper=pd.Series(FEATURE_IMPORTANCE)))
plot_importance(imp, selected, FIG / "feature_importance.png")
plt.show()

## 8. Comparison with the paper

The paper's reported MSE and RMSE are inconsistent with each other (e.g. Grid Search: RMSE 0.238 implies MSE of about 0.057, reported 0.046), so the comparison relies mainly on RMSE, MAE, R2 and MAPE.

In [ ]:
comp = comparison_table(metrics)
display(comp)
display(params_comparison_table(tuned))
plot_metric_comparison(comp, FIG / "paper_vs_ours_metrics.png")
plt.show()

## 9. Stability over different splits

`python scripts/04_seed_robustness.py` repeats split, tuning and test over several seeds; the cell below shows its results once it has been run.

In [ ]:
p = out_dir(cfg, "tables") / "seed_robustness_summary.csv"
if p.exists():
    display(pd.read_csv(p, header=[0, 1], index_col=0))
    from IPython.display import Image
    display(Image(filename=str(FIG / "seed_robustness.png")))
else:
    print("No robustness results yet. Run: python scripts/04_seed_robustness.py")

## 10. Save results to `results/`

In [ ]:
res = {
    "seed": SEED, "n_train": len(X_train), "n_test": len(X_test),
    "tuned": tuned, "metrics": metrics, "importances": imp,
    "selected_by_cv": selected, "y_test": y_test.to_numpy(),
    "preds": {k: np.asarray(v) for k, v in preds.items()},
}
print("Saved to:", save_phase1(res, cfg))